In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Random.fakes import (FakeDIM, FakeRunze, FakePump, FakeGilson, FakeSyrPump)
from Ecosystems.reaction_segment_generation import RSG, RSGState
from Ecosystems.segmentation import (SegmentationState, Segmentation)

In [2]:
# --- Instantiate fake instruments ---
fake_dim = FakeDIM()
fake_runze = FakeRunze()
fake_pump = FakePump()
fake_gilson = FakeGilson()
fake_syrpump = FakeSyrPump()

# --- Create ecosystems ---
rsg = RSG(gilson = fake_gilson, pump = fake_syrpump)
seg = Segmentation(dim=fake_dim, runze=fake_runze, carrier_pump=fake_pump, rsg=rsg)


In [4]:
print(seg.state)

SegmentationState.IDLE


In [3]:
print("\n=== TEST: Segmentation _require_idle ===")
try:
    seg._require_idle()
    print("✅ Segmentation initially IDLE")
except RuntimeError as e:
    print("❌ Failed:", e)

# Force FLOWING to test guard
seg.state = SegmentationState.FLOWING
try:
    seg._require_idle()
except RuntimeError as e:
    print("✅ Caught _require_idle while FLOWING:", e)
seg.state = SegmentationState.IDLE

print("\n=== TEST: Multiple slug launches ===")
for i in range(2):
    print(f"--- Launching slug {i+1} ---")
    fake_dim.load()
    seg.state = SegmentationState.LOOP_LOADED
    seg.launch_segment(flowrate_ul_min=100 + i*20)
    # Simulate segment finish
    seg.finish_flow()
print("✅ Multiple slug launches tested\n")

print("=== TEST: RSG -> Segmentation integration ===")
reaction_plan = [
    {"module": "rack1", "vial": 1, "volume": 5},
    {"module": "rack1", "vial": 2, "volume": 7},
]

meta = rsg.build_reaction(reaction_plan, air_gap_between=1.0)
print("RSG built reaction metadata:", meta)

# Mark loop loaded
seg.mark_loop_loaded()

# Launch via Segmentation
seg.launch_segment(flowrate_ul_min=0.5)
seg.finish_flow()
print("✅ RSG -> Segmentation integration tested\n")

print("=== All ecosystem fake tests completed ===")


=== TEST: Segmentation _require_idle ===
✅ Segmentation initially IDLE
✅ Caught _require_idle while FLOWING: Segmentation is busy or in error state: SegmentationState.FLOWING

=== TEST: Multiple slug launches ===
--- Launching slug 1 ---
[FAKE DIM] Switched to LOAD
[FAKE DIM] Switched to INJECT
[FAKE RUNZE] Moved to position 2
[FAKE PUMP] Flow rate set to 100
[FAKE PUMP] Flow started
[Segmentation] Segment flowing, carrier running at 100 µL/min
[FAKE PUMP] Flow stopped
[Segmentation] Carrier stopped, state = IDLE
--- Launching slug 2 ---
[FAKE DIM] Switched to LOAD
[FAKE DIM] Switched to INJECT
[FAKE RUNZE] Already at position 2
[FAKE PUMP] Flow rate set to 120
[FAKE PUMP] Flow started
[Segmentation] Segment flowing, carrier running at 120 µL/min
[FAKE PUMP] Flow stopped
[Segmentation] Carrier stopped, state = IDLE
✅ Multiple slug launches tested

=== TEST: RSG -> Segmentation integration ===
[Gilson] go_into_vial -> rack1, 1
[Pump] prepare -> rate=0.05, volume=5, dir=WDR
[Pump] start